# LaTeX Perfectionist v25 — Byte Classifier v2 Training

**Model**: CNN+BiLSTM on 128-byte windows (538K params)  
**Target**: F1 ≥ 0.94 on dev set  
**Rules**: 8 ambiguous TYPO rules (deterministic rules handled separately at F1=1.0)  

Run all 7 cells in order. Training resumes automatically if Colab disconnects.

In [ ]:
# Cell 1: Environment + GPU check
!nvidia-smi
import torch
print(f'\nPyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

!pip install -q pyyaml>=6.0 jsonlines>=4.0.0 tqdm>=4.65.0

In [ ]:
# Cell 2: Mount Drive + extract data
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/latex_perfectionist_ml'
os.makedirs(f'{DRIVE_ROOT}/checkpoints_v2', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)

ARCHIVE = 'colab_training_data.tar.gz'
DRIVE_ARCHIVE = f'{DRIVE_ROOT}/{ARCHIVE}'
PROJ = '/content'
os.chdir(PROJ)

if os.path.exists(f'{PROJ}/ml/data/candidates_train.jsonl'):
    print('\u2705 Data already extracted.')
elif os.path.exists(DRIVE_ARCHIVE):
    print('Extracting from Drive...')
    !tar xzf {DRIVE_ARCHIVE} -C {PROJ}/
    print('\u2705 Done.')
else:
    from google.colab import files
    print(f'Upload {ARCHIVE}:')
    uploaded = files.upload()
    if ARCHIVE in uploaded:
        !tar xzf {ARCHIVE} -C {PROJ}/
        !cp {ARCHIVE} {DRIVE_ARCHIVE}
        print('\u2705 Extracted and cached to Drive.')
    else:
        raise RuntimeError(f'Expected {ARCHIVE}, got: {list(uploaded.keys())}')

# Verify required files
required = [
    'ml/data/candidates_train.jsonl', 'ml/data/candidates_dev.jsonl',
    'ml/data/train.jsonl', 'ml/data/dev.jsonl',
    'ml/models/byte_classifier.py', 'ml/models/train_byte_classifier.py',
    'ml/data/candidate_extractor.py', 'ml/config.yaml',
    'specs/rules/vpd_patterns.json',
]
for f in required:
    p = os.path.join(PROJ, f)
    if not os.path.exists(p):
        raise FileNotFoundError(f'Missing: {f}')
    print(f'  \u2705 {f} ({os.path.getsize(p)/1e6:.1f} MB)')
print(f'\n\u2705 All {len(required)} files verified.')

In [ ]:
# Cell 3: Configuration
import sys, os, json, yaml, logging

PROJ = '/content'
sys.path.insert(0, PROJ)
os.chdir(PROJ)

for mod in list(sys.modules):
    if mod.startswith('ml'):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', datefmt='%H:%M:%S')

with open('ml/config.yaml') as f:
    config = yaml.safe_load(f)

import torch

SEED = config['seed']
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = DEVICE == 'cuda'

if DEVICE == 'cuda':
    torch.set_float32_matmul_precision('high')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

EPOCHS = 50
BATCH_SIZE = 256
LR = 1e-3
PATIENCE = 10
MIN_EPOCHS = 5

print(f'Device: {DEVICE}')
print(f'AMP: {USE_AMP}, TF32: {DEVICE == "cuda"}')
print(f'Epochs: {EPOCHS} (patience={PATIENCE}, min={MIN_EPOCHS})')
print(f'Batch: {BATCH_SIZE}, LR: {LR}, Seed: {SEED}')

In [ ]:
# Cell 4: Inspect candidate dataset
import json

for name in ['candidates_train.jsonl', 'candidates_dev.jsonl']:
    path = f'ml/data/{name}'
    total = pos = neg = 0
    rules = {}
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            total += 1
            label = rec.get('label', 0)
            if label == 1: pos += 1
            else: neg += 1
            rid = rec['rule_id']
            if rid not in rules: rules[rid] = {'pos': 0, 'neg': 0}
            rules[rid]['pos' if label else 'neg'] += 1

    print(f'{name}:')
    print(f'  Total: {total:,}  Pos: {pos:,} ({100*pos/total:.1f}%)  Neg: {neg:,} ({100*neg/total:.1f}%)')
    for rid in sorted(rules):
        r = rules[rid]
        print(f'    {rid}: {r["pos"]+r["neg"]:>6,} ({r["pos"]:,} pos, {r["neg"]:,} neg)')
    print()

from ml.data.candidate_extractor import AMBIGUOUS_RULES, DETERMINISTIC_RULES
print(f'Ambiguous (ML): {sorted(AMBIGUOUS_RULES)}')
print(f'Deterministic (F1=1.0): {sorted(DETERMINISTIC_RULES)}')
print(f'\n\u2705 Data ready')

In [ ]:
%%time
# Cell 5: Train ByteClassifier v2
import os, shutil
os.chdir('/content')

from ml.models.train_byte_classifier import train_byte_classifier

DRIVE_V2 = f'{DRIVE_ROOT}/checkpoints_v2'

def on_checkpoint(output_dir, epoch, f1):
    for name in ['best_model.pt', 'training_checkpoint.pt', 'eval_results.json']:
        src = f'{output_dir}/{name}'
        if os.path.exists(src):
            shutil.copy2(src, f'{DRIVE_V2}/{name}')
    print(f'  \u2192 Drive checkpoint (epoch {epoch+1}, F1={f1:.4f})')

RESUME = f'{DRIVE_V2}/training_checkpoint.pt'
resume = RESUME if os.path.exists(RESUME) else None
if resume:
    print('\u267b\ufe0f Resuming from Drive checkpoint')
else:
    print('\U0001f195 Starting fresh')

results = train_byte_classifier(
    train_jsonl='ml/data/candidates_train.jsonl',
    dev_jsonl='ml/data/candidates_dev.jsonl',
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    learning_rate=LR,
    patience=PATIENCE,
    min_epochs=MIN_EPOCHS,
    seed=SEED,
    device=DEVICE,
    output_dir='ml/checkpoints_v2',
    use_amp=USE_AMP,
    save_callback=on_checkpoint,
    resume_from=resume,
)

print(f'\n{"="*60}')
print(f'ByteClassifier v2 Results:')
print(f'  F1:        {results.get("overall_f1", 0):.4f}  (target: >= 0.94)')
print(f'  Precision: {results.get("overall_precision", 0):.4f}')
print(f'  Recall:    {results.get("overall_recall", 0):.4f}')
print(f'  FP rate:   {results.get("measured_fp_rate", 1):.4f}  (target: <= 0.06)')
print(f'  FN rate:   {results.get("measured_fn_rate", 1):.4f}  (target: <= 0.06)')
f1 = results.get('overall_f1', 0)
gate = '\u2705 PASS' if f1 >= 0.94 else '\u274c FAIL'
print(f'  Gate: {gate}')
print(f'{"="*60}')

In [ ]:
# Cell 6: Per-rule breakdown
import pandas as pd

per_rule = results.get('per_rule', {})
if per_rule:
    rows = []
    for rule, m in sorted(per_rule.items()):
        rows.append({
            'Rule': rule,
            'F1': f"{m['f1']:.3f}",
            'Prec': f"{m['precision']:.3f}",
            'Rec': f"{m['recall']:.3f}",
            'TP': m['tp'], 'FP': m['fp'], 'FN': m['fn'], 'TN': m['tn'],
        })
    df = pd.DataFrame(rows)
    print('Per-Rule Results (ambiguous rules only):')
    print(df.to_string(index=False))

    from ml.data.candidate_extractor import DETERMINISTIC_RULES
    print(f'\nDeterministic ({len(DETERMINISTIC_RULES)}): F1=1.000')
    print(f'Ambiguous ({len(per_rule)}): F1={results.get("overall_f1", 0):.4f}')

    weak = [r for r in rows if float(r['F1']) < 0.90]
    if weak:
        print(f'\n\u26a0\ufe0f {len(weak)} rules below 0.90:')
        for r in weak:
            print(f'  {r["Rule"]}: F1={r["F1"]}')
else:
    print('No per-rule breakdown available.')

In [ ]:
# Cell 7: Export Coq bounds + save to Drive + download
import json, os, shutil
from datetime import datetime
os.chdir('/content')

v2_bound = {
    'model': 'ByteClassifier_v2',
    'params': 538625,
    'seed': results.get('seed', 42),
    'threshold': results.get('threshold', 0.5),
    'overall_f1': results.get('overall_f1', 0),
    'overall_precision': results.get('overall_precision', 0),
    'overall_recall': results.get('overall_recall', 0),
    'measured_fp_rate': results.get('measured_fp_rate', 1),
    'measured_fn_rate': results.get('measured_fn_rate', 1),
    'bound_holds': (
        results.get('measured_fp_rate', 1) <= 0.06
        and results.get('measured_fn_rate', 1) <= 0.06
    ),
    'per_rule': results.get('per_rule', {}),
}

os.makedirs('proofs/ML', exist_ok=True)
with open('proofs/ML/v2_eval_bound.json', 'w') as f:
    json.dump(v2_bound, f, indent=2)

print(f'Coq bound:')
print(f'  fp_rate = {v2_bound["measured_fp_rate"]:.4f} (need <= 0.06)')
print(f'  fn_rate = {v2_bound["measured_fn_rate"]:.4f} (need <= 0.06)')
print(f'  bound_holds = {v2_bound["bound_holds"]}')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_dir = f'{DRIVE_ROOT}/runs_v2/{timestamp}'
os.makedirs(run_dir, exist_ok=True)

for src in ['proofs/ML/v2_eval_bound.json',
            'ml/checkpoints_v2/best_model.pt',
            'ml/checkpoints_v2/eval_results.json']:
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(run_dir, os.path.basename(src)))
        print(f'  Saved {os.path.basename(src)} ({os.path.getsize(src)/1e6:.1f} MB)')

print(f'\n\u2705 Saved to Drive: {run_dir}')

# Create download archive
import subprocess
subprocess.run(['tar', 'czf', 'v2_training_results.tar.gz',
                'proofs/ML/v2_eval_bound.json',
                'ml/checkpoints_v2/'], cwd='/content')

print('Downloading results...')
from google.colab import files
files.download('v2_training_results.tar.gz')

## Post-Training

After downloading `v2_training_results.tar.gz`:

```bash
cd /path/to/Scripts
tar xzf v2_training_results.tar.gz
cat proofs/ML/v2_eval_bound.json
# If bound_holds = true, update SpanExtractorSound.v
git add proofs/ML/ ml/checkpoints_v2/
git commit -m "feat(ml): v2 byte classifier — F1=X.XX"
```